In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import numpy as np
import pandas as pd
import random
from sklearn.cluster import SpectralClustering
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
random.seed(42)
np.random.seed(42)

In [ ]:
train_x = np.load("/content/drive/MyDrive/МППО ФИНАЛ/Data.npz")["train_x"]
train_y = np.load("/content/drive/MyDrive/МППО ФИНАЛ/Data.npz")["train_y"]

val_x = np.load("/content/drive/MyDrive/МППО ФИНАЛ/Data.npz")["valid_x"]
val_y = np.load("/content/drive/MyDrive/МППО ФИНАЛ/Data.npz")["valid_y"]

In [ ]:
import numpy as np

def normalize_y_labels(y_array, prefixes=None, sort_classes=True):

    if prefixes is None:
        prefixes = ["K2", "Kepler", "HD", "Gliese", "HIP", "Cancri"]

    def restore_label(label):
        label = str(label)
        positions = []

        for p in prefixes:
            idx = label.find(p)
            if idx != -1:
                positions.append((idx, p))

        if not positions:
            return label

        start_idx = min(positions, key=lambda x: x[0])[0]
        return label[start_idx:start_idx+len(positions[0][1])]

    clean_labels = np.array([restore_label(label) for label in y_array])

    unique_classes = np.unique(clean_labels)
    if sort_classes:
        unique_classes = sorted(unique_classes)

    class_to_idx = {cls: i for i, cls in enumerate(unique_classes)}
    idx_to_class = {i: cls for cls, i in class_to_idx.items()}

    encoded_labels = np.array([class_to_idx[label] for label in clean_labels], dtype=np.int64)

    return clean_labels

In [ ]:
train_y = normalize_y_labels(train_y)
val_y = normalize_y_labels(val_y)

In [ ]:
train_y

array(['Gliese', 'HD', 'K2', ..., 'Gliese', 'K2', 'K2'], dtype='<U6')

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import json


In [ ]:

class AlienSignalDataset(Dataset):
    """Конвертирует сырой waveform в Mel-спектрограмму на лету."""

    def __init__(self, x, y, n_fft=1024, hop_length=512, n_mels=64, augment=False):

        self.waveforms = torch.FloatTensor(x).squeeze(-1)

        if isinstance(y[0], str) or isinstance(y[0], np.str_):
            unique = sorted(set(y))
            self.label_map = {name: i for i, name in enumerate(unique)}
            self.labels = torch.LongTensor([self.label_map[l] for l in y])
            self.num_classes = len(unique)
        else:
            self.labels = torch.LongTensor(y)
            self.num_classes = len(set(y))
            self.label_map = None

        self.augment = augment

        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels

        self.mel_fb = self._create_mel_filterbank(
            sr=16000, n_fft=n_fft, n_mels=n_mels
        )

    def _create_mel_filterbank(self, sr, n_fft, n_mels):
        """Создание мел-фильтров вручную."""
        def hz_to_mel(hz):
            return 2595.0 * np.log10(1.0 + hz / 700.0)

        def mel_to_hz(mel):
            return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

        low_mel = hz_to_mel(0)
        high_mel = hz_to_mel(sr / 2)
        mel_points = np.linspace(low_mel, high_mel, n_mels + 2)
        hz_points = mel_to_hz(mel_points)

        bins = np.floor((n_fft + 1) * hz_points / sr).astype(int)
        n_freqs = n_fft // 2 + 1
        filterbank = np.zeros((n_mels, n_freqs))

        for i in range(n_mels):
            left, center, right = bins[i], bins[i + 1], bins[i + 2]
            for j in range(left, center):
                if center != left:
                    filterbank[i, j] = (j - left) / (center - left)
            for j in range(center, right):
                if right != center:
                    filterbank[i, j] = (right - j) / (right - center)

        return torch.FloatTensor(filterbank)

    def _compute_mel_spectrogram(self, waveform):
        """Waveform → Mel-спектрограмма в dB."""

        window = torch.hann_window(self.n_fft)
        stft = torch.stft(
            waveform, n_fft=self.n_fft, hop_length=self.hop_length,
            win_length=self.n_fft, window=window, return_complex=True
        )
        power = stft.abs() ** 2

        mel = self.mel_fb @ power

        mel = 10.0 * torch.log10(mel + 1e-9)

        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        return mel.unsqueeze(0)

    def _augment(self, mel):
        """Аугментация: маскирование по времени и частоте."""
        _, n_mels, time = mel.shape

        if torch.rand(1) > 0.5:
            f = torch.randint(0, min(8, n_mels), (1,)).item()
            f0 = torch.randint(0, n_mels - f, (1,)).item()
            mel[:, f0:f0 + f, :] = 0

        if torch.rand(1) > 0.5:
            t = torch.randint(0, min(20, time), (1,)).item()
            t0 = torch.randint(0, time - t, (1,)).item()
            mel[:, :, t0:t0 + t] = 0

        if torch.rand(1) > 0.5:
            mel = mel + 0.01 * torch.randn_like(mel)

        return mel

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        mel = self._compute_mel_spectrogram(self.waveforms[idx])
        if self.augment:
            mel = self._augment(mel)
        return mel, self.labels[idx]

In [ ]:

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
        )

    def forward(self, x):
        return self.block(x)


class AlienSignalCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:

def train_model(train_x, train_y, valid_x, valid_y,
                num_epochs=50, batch_size=32, lr=1e-3, device='cuda'):

    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    train_ds = AlienSignalDataset(train_x, train_y, augment=True)
    valid_ds = AlienSignalDataset(valid_x, valid_y, augment=False)

    valid_ds.label_map = train_ds.label_map

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    num_classes = train_ds.num_classes
    print(f"Классов: {num_classes}, Train: {len(train_ds)}, Valid: {len(valid_ds)}")


    model = AlienSignalCNN(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)


    history = {
        'train_loss': [], 'train_acc': [],
        'valid_loss': [], 'valid_acc': [],
    }

    best_valid_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
            total += targets.size(0)

        train_loss = running_loss / total
        train_acc = correct / total


        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in valid_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                running_loss += loss.item() * inputs.size(0)
                correct += (outputs.argmax(1) == targets).sum().item()
                total += targets.size(0)

        valid_loss = running_loss / total
        valid_acc = correct / total

        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['valid_loss'].append(valid_loss)
        history['valid_acc'].append(valid_acc)

        print(f"Epoch {epoch+1:>3}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Valid Loss: {valid_loss:.4f} Acc: {valid_acc:.4f}")

        if valid_acc > best_valid_acc:
            best_valid_acc = valid_acc
            torch.save({
                'model_state_dict': model.state_dict(),
                'label_map': train_ds.label_map,
                'num_classes': num_classes,
                'epoch': epoch,
                'valid_acc': valid_acc,
            }, 'best_model.pth')
            print(f" лучшая сохранена (acc={valid_acc:.4f})")


    with open('training_history.json', 'w') as f:
        json.dump(history, f)


    torch.save({
        'model_state_dict': model.state_dict(),
        'label_map': train_ds.label_map,
        'num_classes': num_classes,
    }, 'final_model.pth')

    print(f"\nЛучшая точность на валидации: {best_valid_acc:.4f}")
    return model, history


data = np.load('/content/drive/MyDrive/МППО ФИНАЛ/Data.npz', allow_pickle=True)
train_x = data['train_x']
train_y = data['train_y']
valid_x = data['valid_x']
valid_y = data['valid_y']

prefixes = ["K2", "Kepler", "HD", "Gliese", "HIP", "Cancri"]

In [ ]:


def normalize_y_labels(y_array, prefixes=None, sort_classes=True):

    if prefixes is None:
        prefixes = ["K2", "Kepler", "HD", "Gliese", "HIP", "Cancri"]

    def restore_label(label):
        label = str(label)
        positions = []

        for p in prefixes:
            idx = label.find(p)
            if idx != -1:
                positions.append((idx, p))

        if not positions:
            return label

        start_idx = min(positions, key=lambda x: x[0])[0]
        return label[start_idx:start_idx+len(positions[0][1])]

    clean_labels = np.array([restore_label(label) for label in y_array])

    unique_classes = np.unique(clean_labels)
    if sort_classes:
        unique_classes = sorted(unique_classes)

    class_to_idx = {cls: i for i, cls in enumerate(unique_classes)}
    idx_to_class = {i: cls for cls, i in class_to_idx.items()}

    encoded_labels = np.array([class_to_idx[label] for label in clean_labels], dtype=np.int64)

    return clean_labels

train_y_clean = normalize_y_labels(train_y)
valid_y_clean = normalize_y_labels(val_y)

print(f"train_x shape: {train_x.shape}")
print(f"Классы: {sorted(set(train_y_clean))}")

model, history = train_model(
    train_x, train_y_clean,
    valid_x, valid_y_clean,
    num_epochs=5,
    batch_size=32,
    lr=1e-3,
)


